In [24]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

import utils
importlib.reload(utils)
from utils import *

import config
importlib.reload(config)
from config import *

##### Before Running:
---
- Update the data_dir in config.py, make sure there are separate files for each participant with all the data belonging to each of them
- Change  S_TIMINGS_study_date_1 if the station timings are different, or add a new day with new timings if necessary
- IF this is the case, add the new day timings to S_TIMINGS and the date and corresponding participant IDs in the groups
---

In [ ]:
# Per-participant metadata : things that can't be auto-detected from filenames
participant_meta = {}
PARTICIPANT_MAP = {}
PARTICIPANT_MAP_2 = {}
study_groups = {}

for date_str, (pids, timings, shifts) in groups.items():

    compact_date = date_str.replace("-", "")
    study_groups[compact_date] = pids
    
    for pid in pids:
        # Build participant metadata
        participant_meta[pid] = {
            "date": date_str,
            "start": start,
            "end": end,
            "actlumus_shift_min": shifts[0],
            "actlumus_shift_sec": shifts[1]
        }
        # Maps based on timings existence (1-2)
        if timings is not None:
            PARTICIPANT_MAP[pid] = (date_str, timings)
        else:
            PARTICIPANT_MAP_2[pid] = date_str


In [ ]:
os.makedirs(cumulative_out, exist_ok=True)

In [ ]:
participant_studydate_w_stations = pd.DataFrame(PARTICIPANT_MAP).T.reset_index()
participant_studydate_w_stations.columns = ["participant_id", "date", "stations"]
participant_studydate_w_stations = participant_studydate_w_stations.sort_values(by='participant_id', ignore_index=True)

participant_studydate_without_stations = pd.DataFrame(list(PARTICIPANT_MAP_2.items()),columns=["participant_id", "date"])
participant_studydate_without_stations.columns = ["participant_id", "date"]
participant_studydate_without_stations = participant_studydate_without_stations.sort_values(by='participant_id', ignore_index=True)

participant_studydate_w_stations["has_stations"] = True
participant_studydate_without_stations["has_stations"] = False

participant_studydate = pd.concat(
    [participant_studydate_w_stations, participant_studydate_without_stations],
    ignore_index=True
)

In [ ]:
#--------------------------------------------------------------------------
# DETECT FILES FOR EACH PARTICIPANT
# Print a summary for verification before plotting
#--------------------------------------------------------------------------

print("=" * 60)
print("FILE DETECTION SUMMARY")
print("=" * 60)

all_detected = {}
for study_date, participants in study_groups.items():
    print(f"\n--- Study date: {study_date} ---")
    for pid in participants:
        if pid not in participant_meta:
            print(f"  {pid}: !! MISSING from participant_meta")
            continue
        meta = participant_meta[pid]
        detected = detect_files(pid, meta, data_dir)
        all_detected[pid] = detected

        print(f"\n  {pid}:")
        for ftype, fpath in detected.items():
            if fpath:
                print(f"    {ftype:15s} ✓  {os.path.relpath(fpath, data_dir)}")
            else:
                print(f"    {ftype:15s} ✗  NOT FOUND")

print("\n" + "=" * 60)
print("Review the above before running the plots section.")
print("=" * 60)

data_dict = all_detected 

---
#### CUMULATIVE
---

In [22]:
custom_hex = [
"#FF0000", # Vivid Red
"#00FF00", # Bright Green
"#0000FF", # Royal Blue
"#FF00FF", # Magenta
"#FF8000", # Bright Orange
"#00FFFF", # Cyan/Aqua
"#800000", # Maroon/Dark Red
"#008080"  # Teal
]

In [ ]:
# MAIN LOOP PER DATE 
#--------------------------------------------------------------------------
for date_str, (p_list, timing_key, shift) in groups.items():
    # Initialize all 8 Figures
    fig_hr, ax_hr = plt.subplots(figsize=(14, 10))
    fig_rr, ax_rr = plt.subplots(figsize=(14, 10))
    fig_light, ax_light = plt.subplots(figsize=(14, 7))
    fig_ir, ax_ir = plt.subplots(figsize=(14, 7))
    fig_mel, ax_mel = plt.subplots(figsize=(14, 7))
    fig_act_temp, ax_act_temp = plt.subplots(figsize=(14, 7))
    fig_hand, ax_hand = plt.subplots(figsize=(14, 7))
    fig_chest, ax_chest = plt.subplots(figsize=(14, 7))
    
    start_dt = pd.to_datetime(f"{date_str} {start}")
    end_dt = pd.to_datetime(f"{date_str} {end}")

    for idx, p_id in enumerate(p_list):
        files = data_dict.get(p_id)
        if not files: continue
        
        # preparation
        m = participant_meta.get(p_id, {"actlumus_shift_min": 0, "actlumus_shift_sec": 0})
        p_shift = pd.Timedelta(minutes=m["actlumus_shift_min"], seconds=m["actlumus_shift_sec"])
        current_color = custom_hex[idx % len(custom_hex)]
        line_params = {'color': current_color, 'label': p_id, 'linewidth': 0.5, 'alpha': 0.9}

        # ActLumus (Light, IR, Mel, Temp)
        if files.get('actlumus'):
            try:
                skip = get_actlumus_start(files['actlumus'])
                act = pd.read_csv(files['actlumus'], sep=';', skiprows=skip, header=0, engine='python')
                act.columns = [c.strip().upper() for c in act.columns]
                time_col = next((c for c in act.columns if 'DATE' in c and 'TIME' in c), None)
                
                if time_col:
                    act['DT_CLEAN'] = pd.to_datetime(act[time_col], dayfirst=True, format='mixed', errors='coerce') - p_shift
                    act = act.dropna(subset=['DT_CLEAN'])
                    act_p = act[(act['DT_CLEAN'] >= start_dt) & (act['DT_CLEAN'] <= end_dt)]
                    
                    if not act_p.empty:
                        if 'LIGHT' in act_p.columns: ax_light.plot(act_p['DT_CLEAN'], act_p['LIGHT'], **line_params)
                        ir_col = next((c for c in act_p.columns if 'IR' in c or 'INFRA' in c), None)
                        if ir_col: ax_ir.plot(act_p['DT_CLEAN'], act_p[ir_col], **line_params)
                        mel_col = next((c for c in act_p.columns if 'MEL' in c or 'MEDI' in c), None)
                        if mel_col: ax_mel.plot(act_p['DT_CLEAN'], act_p[mel_col], **line_params)
                        t_col = next((c for c in act_p.columns if 'TEMP' in c and 'UNIT' not in c), None)
                        if t_col: ax_act_temp.plot(act_p['DT_CLEAN'], act_p[t_col], **line_params)
            except Exception as e:
                print(f"Error ActLumus {p_id}: {e}")

        #HR & HRV
        if files.get('hr'):
            try:
                df_hr = pd.read_csv(files['hr'])
                meta = df_hr.iloc[0]
                start_time = pd.to_datetime(f"{meta['Date']} {meta['Start time']}", dayfirst=True, format='mixed')
                
                # Plot Heart Rate
                hr_c = df_hr.iloc[2:].copy()
                hr_c.columns = [str(c).strip() for c in df_hr.iloc[1]]
                hr_c["ts"] = start_time + pd.to_timedelta(hr_c["Time"].astype(str))
                hr_p = hr_c[(hr_c["ts"] >= start_dt) & (hr_c["ts"] <= end_dt)]
                if not hr_p.empty:
                    ax_hr.plot(hr_p["ts"], pd.to_numeric(hr_p['HR (bpm)'], errors='coerce'), **line_params)
                
                # Plot HRV (With Preprocessing)
                if files.get('hrv'):
                    df_rr = pd.read_csv(files['hrv'])
                    df_rr['ts'] = start_time + pd.to_timedelta(df_rr['duration'].cumsum(), unit='ms')
                    rr_p = df_rr[(df_rr['ts'] >= start_dt) & (df_rr['ts'] <= end_dt)].copy()

                    if not rr_p.empty:
                        rr_series = rr_p.set_index('ts')['duration']
                        rr_clean = preprocess_rr(
                            rr_series,
                            plausible_min=00,  #add preprocessing parameters here
                            plausible_max=50000,
                            jump_threshold=00,
                            smooth_window=1
                        )
                        ax_rr.plot(rr_clean.index, rr_clean.values, **line_params)
            except Exception as e:
                print(f"Error HR/HRV {p_id}: {e}")

        #iButtons
        for key, ax_ib in [('ibutton_hand', ax_hand), ('ibutton_chest', ax_chest)]:
            if files.get(key):
                try:
                    skip = get_ibutton_start(files[key])
                    ib = pd.read_csv(files[key], sep=',', header=None, names=["D","T","U","V","Dec"], skiprows=skip, engine='python')
                    ib["temp"] = pd.to_numeric(ib['V'], errors='coerce') + pd.to_numeric(ib['Dec'], errors='coerce') / 1000
                    ib["dt"] = pd.to_datetime(ib["D"] + " " + ib["T"], dayfirst=True, format='mixed', errors='coerce')
                    ib_p = ib[(ib['dt'] >= start_dt) & (ib['dt'] <= end_dt)]
                    if not ib_p.empty:
                        ax_ib.plot(ib_p['dt'], ib_p['temp'], **line_params)
                except Exception as e: print(f"Error iButton {p_id}: {e}")

    #FORMAT & SAVE
    plot_list = [
        (ax_hr, "Heart Rate", "BPM", "HR"), 
        (ax_rr, "HRV (RR Intervals)", "ms", "RR_preprocessed"),
        (ax_light, "Light", "Lux", "Light"), 
        (ax_ir, "IR", "Counts", "IR"),
        (ax_mel, "Melanopic EDI", "lx", "Melanopic"), 
        (ax_act_temp, "ActLumus Temp", "°C", "ActTemp"),
        (ax_hand, "Hand Temp", "°C", "Hand"), 
        (ax_chest, "Chest Temp", "°C", "Chest")
    ]

    for ax, title, ylabel, fname in plot_list:
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            # Add Shading
            if timing_key and timing_key in S_TIMINGS:
                for s_start, s_end in S_TIMINGS[timing_key].values():
                    ax.axvspan(pd.to_datetime(f"{date_str} {s_start}"), 
                               pd.to_datetime(f"{date_str} {s_end}"), color="gray", alpha=0.1)
            
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.set_title(f"{title} - {date_str}")
            ax.set_ylabel(ylabel)
            ax.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='7')
            plt.setp(ax.get_xticklabels(), rotation=45)
            
            fig = ax.get_figure()
            fig.savefig(os.path.join(cumulative_out, f"Cumulative_{fname}_{date_str}.png"), dpi=400, bbox_inches='tight')
        
        plt.close(ax.get_figure())